# Step 11 — Fast competitive tabular IDS results (not more CNN diagnosis)

**Research goal:** obtain real, competitive ten-class UNSW-NB15 macro-F1 quickly before attempting a novel model. We will compare two LightGBM boosted-tree models on the *same* exact 35,005 validation target rows used by V0 CNN–ViT (55.87% macro-F1), V1 ViT-only (57.44%), V2 CNN–ViT repair (55.25%).

**The proposed model novelty is not claimed here.** This is a strong baseline and a practical sanity check of whether 16×16 RGB traffic image serialization is helping. All previous model weights and predictions remain untouched. Official UNSW test remains completely unused.

- **Features**: the original 42 network-flow columns, including proto/service/state categories. No `id`, `label` (binary attack label), or `attack_cat` label included as input.
- **Data**: original train-only group-disjoint split; boosted models fit on exactly 140,328 original 5-row-window *target rows*, predict exactly the same 35,005 validation target labels. No arbitrary CSV adjacency or image encoding used.
- **Models**: LightGBM multiclass unweighted; and square-root class-balanced LightGBM. Both seed 42; stop at maximum *internal validation macro-F1*. Model selection uses internal validation, so this is **exploratory model development**, not unbiased held-out evaluation.
- **Progress and outputs**: per-tree progress bars, measured scores printed immediately after each model, all 10-class confusion counts and per-class F1, full probability arrays saved privately, cryptographic model hashes, observed figures from numeric sources. This run is intended to be much faster than a 50-epoch ViT, but actual time depends on Kaggle CPU.
- **Safety**: use only original designated *training* CSV, verify SHA-256, exact original original split, class ordering, and frozen validation targets. Don't claim our scores are directly comparable to Chen et al.'s published official benchmark.

**Kaggle:** Use the existing Kaggle session if still open and saved research_outputs are present; otherwise enable Internet, authorize `pushRare` to restore original archive, and attach `dsshayan/unsw-nb15`. A GPU accelerator is **not required** for LightGBM. No extra figures or CNN diagnostics to run first.

In [ ]:
# Single Kaggle cell — real fast boosted-tree model training and results.
from pathlib import Path
import subprocess, sys, os, importlib.util

REPO=Path("/kaggle/working/rare-attack-ids-fast-tabular")
BASE=Path("/kaggle/working/research_outputs")
RUN=BASE/"CTRL-TAB-001-S42"
ARCHIVE=Path("/kaggle/working/SCV-IDS-REPRO-001_FULL_ARTIFACTS.tar")
DATA=Path("/kaggle/input/datasets/dsshayan/unsw-nb15/UNSW_NB15_training-set.csv")
assert DATA.is_file(), "Attach original Kaggle dataset dsshayan/unsw-nb15."
url="https://github.com/itsCodeBakery/rare-attack-spatiotemporal-ids.git"
if REPO.is_dir():
    subprocess.run(["git","-C",str(REPO),"pull","--ff-only"],check=True)
else:
    subprocess.run(["git","clone","--depth","1",url,str(REPO)],check=True)
print("Research commit:",subprocess.check_output(
    ["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip())
if importlib.util.find_spec("lightgbm") is None:
    subprocess.run([sys.executable,"-m","pip","install","-q","lightgbm>=4.0,<5"],check=True)

env=os.environ.copy()
env["PYTHONPATH"]=str(REPO/"src")
env["MPLBACKEND"]="Agg"
env["OMP_NUM_THREADS"]="4"
env["OPENBLAS_NUM_THREADS"]="1"

# Restore frozen research data only if the prior Kaggle session expired.
required=[BASE/"SCV-IDS-REPRO-001"/"split_indices.npz",
          BASE/"SCV-IDS-REPRO-001"/"preprocessing.json",
          BASE/"SCV-IDS-REPRO-001"/"validation_predictions.npz"]
if not all(p.is_file() for p in required):
    from kaggle_secrets import UserSecretsClient
    key=UserSecretsClient().get_secret("pushRare")
    secure=env.copy(); secure["GITHUB_TOKEN"]=key
    try:
        subprocess.run([sys.executable,
           str(REPO/"scripts"/"restore_scv_release.py"),
           "--base",str(BASE),"--archive",str(ARCHIVE)],
           cwd=str(REPO),env=secure,check=True)
    finally:
        secure.pop("GITHUB_TOKEN",None)
        del key

# Checkpoints and predictions go into NEW directory; original experiments untouched.
subprocess.run([
    sys.executable,str(REPO/"scripts"/"run_ctrl003_tabular_fast.py"),
    "--repo",str(REPO),
    "--config",str(REPO/"configs"/"ctrl003_fast_tabular.json"),
    "--reference",str(BASE/"SCV-IDS-REPRO-001"),
    "--output",str(RUN)
],cwd=str(REPO),env=env,check=True)

print("REAL MEASURED SCORES:")
print((RUN/"aggregate_model_scores.csv").read_text(encoding="utf-8"))
print("Model weights, raw probability predictions, confusion and plots:",
      str(RUN))

### Return these results before proceeding

Send the terminal output under **`FAST TABULAR REAL RESULTS COMPLETE`**, particularly macro-F1, accuracy, and balanced accuracy for both models.

This first screen **does not** retrain any CNN or ViT. We'll choose whether to invest additional Kaggle GPU time in CatBoost or a novel confusion-aware rare-attack method *after* seeing genuine boosted-tree results. Save the notebook version if leaving Kaggle; the full model files and private prediction arrays must be retained before session expiry.